[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **AI Powered Hand Gesture Interaction Pipeline**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog/<BLOG_NAME>)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)

## Project Overview  
This project is a low-latency Human-Computer Interaction (HCI) system that converts real-time hand gestures from a webcam into precise OS-level controls such as:
- Mouse movement  
- Mouse clicks  
- Keyboard shortcuts  
- Media controls  

Unlike traditional systems built on fixed frameworks like MediaPipe, this project uses a fully custom-trained keypoint detection pipeline powered by structural annotations exported from Labellerr.

---

## Core Architecture  

### Custom Data Collection  
Captured diverse hand movement videos under different lighting conditions, perspectives, and hand poses.

### Precision Labellerr Annotation  
Annotated a custom 21-keypoint skeletal structure using the Labellerr Keypoint Annotation Platform.

### Custom Model Training  
Trained a lightweight deep learning / ML-based keypoint prediction model using exported structural datasets.

### Smooth OS Translation Layer  
Applied an Exponential Moving Average (EMA) filter to stabilize predictions before sending them to OS controls via PyAutoGUI.

---

## Key Advantages  

- Higher localization accuracy  
- Better adaptability to real-world environments  
- Improved robustness against occlusions  
- Independence from generic public hand-tracking frameworks  

---

## Real-World Applications  

### Sterile Medical Environments  
Enables touch-free interaction with medical imaging systems during surgeries.

### Smart Manufacturing  
Allows operators to control industrial systems using gestures, even with gloves or difficult lighting.

### Assistive Technology  
Provides customizable gesture-based computer access for users with mobility impairments.

### Smart Classrooms & Presentations  
Enables presenters to control slides and digital displays without physical remotes.

---

## Annotate your Custom dataset using Labellerr

 ***1. Visit the [Labellerr](https://www.labellerr.com/?utm_source=githubY&utm_medium=social&utm_campaign=github_clicks) website and click **“Sign Up”**.*** 

 ***2. After signing in, create your workspace by entering a unique name.***

 ***3. Navigate to your workspace’s API keys page (e.g., `https://<your-workspace>.labellerr.com/workspace/api-keys`) to generate your **API Key** and **API Secret**.***

 ***4. Store the credentials securely, and then use them to initialise the SDK or API client with `api_key`, `api_secret`.*** 



## Import Libraries

This section imports all the required libraries used throughout the project for computer vision, visualization, deep learning, and structured coding.


In [1]:
!git clone https://github.com/Labellerr/yolo_finetune_utils.git

Cloning into 'yolo_finetune_utils'...


In [1]:
!pip install ultralytics opencv-python matplotlib cv2

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for cv2


In [2]:
!pip install --user --no-cache-dir mediapipe opencv-python pyautogui


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [4]:
!pip uninstall -y pyautogui
!pip cache purge
!pip install pyautogui

Found existing installation: PyAutoGUI 0.9.54
Uninstalling PyAutoGUI-0.9.54:
  Successfully uninstalled PyAutoGUI-0.9.54
Files removed: 2 (2.0 kB)
Directories removed: 0
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for pyautogui: filename=pyautogui-0.9.54-py3-none-any.whl size=37707 sha256=92129548e8f35184fccf68a10402ed4c8d4d22cf8c5e0206e98000359a990b17
  Stored in directory: c:\users\msi msi\appdata\local\pip\cache\wheels\9c\22\e7\c97f656d0790ef69feae4fcc7bbc1fc648f1b37879f52e7480
Successfully built pyautogui



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


## 🎞️ Random Frame Extraction from Video

Extracts a fixed number of high-quality frames from one or more videos to create an image dataset for annotation and training.

### 🔹 Purpose
- Convert raw manufacturing videos into individual image frames  
- Perform random sampling to avoid frame bias  
- Prepare data for annotation and YOLO training  


In [ ]:
%pip install tqdm

from yolo_finetune_utils.frame_extractor import extract_random_frames

extract_random_frames(
    paths=['WIN_20260519_12_17_01_Pro.mp4'],
    total_images=50,
    out_dir="dataset_Frames",
    jpg_quality=100,
    seed=42
)


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[✓] Extracted 5 frames to folder: dataset_Frames


## 📥 Download Annotations from Labellerr

After completing data labeling on the **Labellerr** platform, export the annotations in **COCO JSON format**.

Download the COCO JSON file from the Labellerr website and upload it into this project workspace to use it for further dataset preparation and training.

This COCO JSON file will be used in the next steps for:
- Frame–annotation alignment
- COCO → YOLO format conversion
- Model training and evaluation


# COCO to YOLO Format Conversion

Converts COCO-style segmentation annotations to YOLO segmentation dataset format.  
- Requires: `annotation.json` and images in `frames_output` directory.
- Output: Generated YOLO dataset folder.
- Parameters: allows train/val split, shuffling, and verbose mode.


In [17]:
import json
import os
import shutil
import random
from pathlib import Path

# --- CONFIGURATION ---
base_path = r'D:\Desktop\Desk\Labellerr Github Projects\Use_Case_Projects\hand gesture egocentric'
json_path = os.path.join(base_path, 'export-#fa3d8e71-fc68-499b-bc5d-47fef226a735.json')
images_src = os.path.join(base_path, 'dataset_Frames')
output_path = os.path.join(base_path, 'yolo_dataset')

# Ratios for your 30 frames
ratios = {'train': 0.8, 'val': 0.1, 'test': 0.1}

# Create Folder Structure
for split in ratios.keys():
    os.makedirs(os.path.join(output_path, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_path, 'labels', split), exist_ok=True)

# Generate data.yaml (formatted for YOLO11-Pose)
formatted_path = output_path.replace('\\', '/')
yaml_content = f"""
path: {formatted_path}
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: hand

kpt_shape: [21, 3]
"""
with open(os.path.join(base_path, 'data.yaml'), 'w') as f:
    f.write(yaml_content.strip())

# Load COCO JSON
with open(json_path, 'r') as f:
    coco_data = json.load(f)

# Build category map (Ensures 0: bottle, 1: cap, 2: Hands)
cat_map = {}
for cat in coco_data['categories']:
    name = cat['name'].lower()
    if 'bottle' in name: cat_map[cat['id']] = 0
    elif 'cap' in name: cat_map[cat['id']] = 1
    elif 'hand' in name: cat_map[cat['id']] = 2

images = {img['id']: img for img in coco_data['images']}
annotations = coco_data['annotations']

# Shuffle and Split IDs
img_ids = list(images.keys())
random.seed(42)
random.shuffle(img_ids)

train_idx = int(len(img_ids) * ratios['train'])
val_idx = train_idx + int(len(img_ids) * ratios['val'])

def convert_to_yolo(ann, img_w, img_h):
    cls_id = cat_map.get(ann['category_id'], -1)
    if cls_id == -1: return None

    line = [str(cls_id)]
    has_data = False
    
    # 1. Add Segmentation (Polygons for all classes)
    if 'segmentation' in ann and ann['segmentation']:
        poly = ann['segmentation'][0]
        if len(poly) > 4:
            for i in range(0, len(poly), 2):
                line.append(f"{poly[i]/img_w:.6f}")
                line.append(f"{poly[i+1]/img_h:.6f}")
            has_data = True
            
    # 2. Add Keypoints (ONLY for Hands class - ID 2)
    if cls_id == 2 and 'keypoints' in ann and ann['keypoints']:
        kpts = ann['keypoints']
        for i in range(0, len(kpts), 3):
            line.append(f"{kpts[i]/img_w:.6f}")
            line.append(f"{kpts[i+1]/img_h:.6f}")
            line.append(str(kpts[i+2]))
        has_data = True
            
    return " ".join(line) if has_data else None

# Process Files
for i, img_id in enumerate(img_ids):
    img_info = images[img_id]
    filename = img_info['file_name']
    split = 'train' if i < train_idx else 'val' if i < val_idx else 'test'
    
    src_img = os.path.join(images_src, filename)
    if os.path.exists(src_img):
        shutil.copy(src_img, os.path.join(output_path, 'images', split, filename))
        
        label_name = Path(filename).stem + ".txt"
        with open(os.path.join(output_path, 'labels', split, label_name), 'w') as lf:
            for ann in annotations:
                if ann['image_id'] == img_id:
                    yolo_line = convert_to_yolo(ann, img_info['width'], img_info['height'])
                    if yolo_line:
                        lf.write(yolo_line + "\n")

print(f"Conversion Done. Dataset and data.yaml are ready at: {base_path}")

Conversion Done. Dataset and data.yaml are ready at: D:\Desktop\Desk\Labellerr Github Projects\Use_Case_Projects\hand gesture egocentric


# Load and Train YOLO Pose Model

Loads the YOLO segmentation model and trains it using the converted YOLO dataset.
- Data: Path to YOLO-style `data.yaml`
- Parameters: epochs, image size, batch size, device, dataloader workers, experiment name.


In [ ]:
from ultralytics import YOLO

# Using the x-pose model for your 21-point hand skeleton
model = YOLO('yolo11l-pose.pt') 

model.train(
    data='/kaggle/working/ydata.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    project='/kaggle/working/factory',
    name='train_run_11',
    device=0
)

# Custom Gesture Controller Architecture

## The 4-Step Pipeline

```text
[Webcam Frame]
      │
      ▼
[Custom AI Model]
      │  → Runs inference and outputs a (21,3) keypoint array
      ▼
[MockLandmark Wrapper]
      │  → Converts raw coordinates into (.x, .y) landmark objects
      ▼
[Gesture Logic Engine]
      │  → Detects active gestures using finger-state comparisons
      ▼
[PyAutoGUI Execution]
         → Executes mouse and keyboard actions

In [ ]:
import cv2
import pyautogui
import math
import numpy as np
# import torch # Uncomment if using a PyTorch model (.pt)
# import joblib # Uncomment if using a Scikit-Learn model (.pkl)

# --- CUSTOM MODEL INITIALIZATION PLACEHOLDER ---
# Replace this section with your actual model loading syntax.
def load_custom_model():
    # Example for YOLO / PyTorch / ONNX model:
    # model = torch.load('custom_labellerr_keypoints.pt')
    # return model
    print("Custom Labellerr-trained keypoint model loaded successfully.")
    return None

my_keypoint_model = load_custom_model()

def predict_keypoints(frame, model):
    """
    Dummy inference wrapper. Replace this logic with your model's forward pass.
    Should return a NumPy array of shape (21, 2) with normalized x, y coordinates [0.0, 1.0].
    """
    # 1. Preprocess frame to match your model input size (e.g., resize, normalize)
    # img = cv2.resize(frame, (224, 224)) / 255.0
    
    # 2. Run inference
    # predictions = model(img)
    
    # For now, we return a structural placeholder array to prevent code crashes
    return np.zeros((21, 2)) 

# --- MOCK CLASS TO KEEP YOUR EXISTING LOGIC SYNTAX INTACT ---
class MockLandmark:
    def __init__(self, x, y):
        self.x = x
        self.y = y

# Setup screen metrics for PyAutoGUI
screen_width, screen_height = pyautogui.size()
pyautogui.FAILSAFE = True 

# --- SMOOTHING CONFIGURATION ---
alpha = 0.25  
prev_x, prev_y = 0, 0

# --- STATE LOCKS (DEBOUNCE FLAGS) ---
media_flag = False

cap = cv2.VideoCapture(0)

try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            continue

        frame = cv2.flip(frame, 1)
        h, w, c = frame.shape
        
        # --- INFREENCE VIA YOUR CUSTOM MODEL ---
        # Completely bypassing results = hands.process(rgb_frame)
        predictions = predict_keypoints(frame, my_keypoint_model)
        
        # Ensure a hand was actually detected before running calculations
        hand_detected = True # Set your model's object detection/confidence check here
        
        if hand_detected:
            # Reconstruct the 21 coordinates to index exactly like your Labellerr topology
            landmarks = [MockLandmark(pt[0], pt[1]) for pt in predictions]
            
            # --- THE REST OF YOUR LOGIC STAYS 100% UNCHANGED ---
            # Tips (Indices map exactly to your Labellerr layout names)
            thumb_tip = landmarks[4]
            index_tip = landmarks[8]
            middle_tip = landmarks[12]
            ring_tip = landmarks[16]
            pinky_tip = landmarks[20]
            
            # Knuckles (MCP joints)
            index_mcp = landmarks[5]
            middle_mcp = landmarks[9]
            ring_mcp = landmarks[13]
            pinky_mcp = landmarks[17]

            # Optional: Draw custom connections manually using your Labellerr map
            for lm in landmarks:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 4, (0, 255, 255), cv2.FILLED)

            # Check finger extension states
            index_up = index_tip.y < index_mcp.y
            middle_up = middle_tip.y < middle_mcp.y
            ring_up = ring_tip.y < ring_mcp.y
            pinky_up = pinky_tip.y < pinky_mcp.y

            ix, iy = int(index_tip.x * w), int(index_tip.y * h)
            tx, ty = int(thumb_tip.x * w), int(thumb_tip.y * h)
            click_distance = math.hypot(tx - ix, ty - iy)

            # ---------------- GESTURE PROCESSING ----------------
            
            # GESTURE A: Open Palm -> Media Play / Pause
            if index_up and middle_up and ring_up and pinky_up:
                cv2.putText(frame, "PLAY/PAUSE DETECTED", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
                if not media_flag:
                    pyautogui.press('playpause', _pause=False)
                    media_flag = True
            
            # GESTURE B: Three Fingers Up -> Volume Down
            elif index_up and middle_up and ring_up and not pinky_up:
                cv2.putText(frame, "VOLUME DOWN", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                pyautogui.press('volumedown', _pause=False)
                media_flag = False 

            # GESTURE C: Two Fingers Up -> Volume Up
            elif index_up and middle_up and not ring_up and not pinky_up:
                cv2.putText(frame, "VOLUME UP", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                pyautogui.press('volumeup', _pause=False)
                media_flag = False 

            # GESTURE D: Only Index Finger Up -> Mouse Control
            elif index_up and not middle_up and not ring_up and not pinky_up:
                cv2.putText(frame, "MOUSE MODE", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                media_flag = False 
                
                target_x = int(index_tip.x * screen_width)
                target_y = int(index_tip.y * screen_height)
                
                # Smooth cursor adjustment
                smoothed_x = int(alpha * target_x + (1 - alpha) * prev_x)
                smoothed_y = int(alpha * target_y + (1 - alpha) * prev_y)
                
                pyautogui.moveTo(smoothed_x, smoothed_y, _pause=False)
                prev_x, prev_y = smoothed_x, smoothed_y

                cv2.line(frame, (ix, iy), (tx, ty), (255, 0, 0), 2)
                if click_distance < 32: 
                    cv2.circle(frame, (ix, iy), 12, (0, 255, 0), cv2.FILLED)
                    pyautogui.click(_pause=False)
                    cv2.waitKey(200) 

            else:
                media_flag = False

        cv2.imshow('Labellerr Custom Trained Controller', frame)

        if cv2.waitKey(1) & 0xFF == ord('q') or cv2.getWindowProperty('Labellerr Custom Trained Controller', cv2.WND_PROP_VISIBLE) < 1:
            break
            
finally:
    cap.release()
    cv2.destroyAllWindows()
    print("System Closed Safely.")

---

## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*
